In [ ]:
import pandas as pd

# 6.2
df = pd.read_excel(
    "dukes_raw/DUKES_6.2_Capacity of, generation from renewable sources and shares of total generation.xlsx",
    sheet_name="6.2",
    header=None
)

print("=== 6.2 head(10) ===")
print(df.head(10))
print("\n=== 6.2 no.0 ===")
print(df.iloc[0])

In [ ]:
# 6.3
df = pd.read_excel(
    "dukes_raw/DUKES_6.3_Load factors for renewable electricity generation.xlsx",
    sheet_name="6.3",
    header=None
)

print("=== 6.3 head(10) ===")
print(df.head(10))
print("\n=== 6.3 no.0 ===")
print(df.iloc[0])

In [ ]:
import pandas as pd
import os

FILE_62 = "dukes_raw/DUKES_6.2_Capacity of, generation from renewable sources and shares of total generation.xlsx"
SHEET_62 = "6.2"

raw_62 = pd.read_excel(FILE_62, sheet_name=SHEET_62, header=None)

def extract_62_section(raw, section_title, value_name, source_table):
    header_idx = raw.index[raw.iloc[:, 0].astype(str).str.strip().eq(section_title)].min()
    assert pd.notna(header_idx), f"Could not find section: {section_title}"

    following_blank = raw.index[
        (raw.index > header_idx) & (raw.iloc[:, 0].isna())
    ].min()
    assert pd.notna(following_blank), f"Could not find end of section: {section_title}"

    section = raw.iloc[header_idx + 1:following_blank].copy()
    section.columns = raw.iloc[header_idx].tolist()
    section = section.rename(columns={section.columns[0]: "source_tech"})

    rename_year_cols = {}
    year_cols = []

    for col in section.columns:
        clean = str(col).split("[")[0].replace(".0", "").strip()
        if clean.isdigit():
            y = int(clean)
            rename_year_cols[col] = y
            if 2010 <= y <= 2024:
                year_cols.append(y)

    section = section.rename(columns=rename_year_cols)

    out = section.melt(
        id_vars=["source_tech"],
        value_vars=year_cols,
        var_name="year",
        value_name=value_name
    )

    out["year"] = out["year"].astype(int)
    out[value_name] = pd.to_numeric(out[value_name], errors="coerce")
    out["source_tech"] = out["source_tech"].astype(str).str.strip()
    out = out.dropna(subset=[value_name]).copy()
    out["source_table"] = source_table

    return out

df_62_capacity_context = extract_62_section(
    raw_62,
    "Installed Capacity (MW) [note 1]",
    "capacity_mw",
    "DUKES 6.2 installed capacity"
)

df_62_generation_context = extract_62_section(
    raw_62,
    "Generation (GWh) [note 8]",
    "generation_gwh",
    "DUKES 6.2 generation"
)

os.makedirs("output", exist_ok=True)

df_62_capacity_context.to_parquet(
    "output/dukes_renewables_capacity_context_2010_2024.parquet",
    index=False
)

df_62_generation_context.to_parquet(
    "output/dukes_renewables_generation_context_2010_2024.parquet",
    index=False
)

print("6.2 context extraction complete")
print("Capacity context rows:", len(df_62_capacity_context))
print("Generation context rows:", len(df_62_generation_context))
print("These are saved as context files and are not appended to dukes_capacity_hist.")

In [ ]:
import pandas as pd
import os

FILE_63 = "dukes_raw/DUKES_6.3_Load factors for renewable electricity generation.xlsx"
SHEET_63 = "6.3"

raw_63 = pd.read_excel(FILE_63, sheet_name=SHEET_63, header=None)

header_idx = raw_63.index[
    raw_63.iloc[:, 0].astype(str).str.contains(
        "Load factors - based on average of beginning and end of year capacity",
        na=False
    )
].min()

next_section_idx = raw_63.index[
    (raw_63.index > header_idx) &
    raw_63.iloc[:, 0].astype(str).str.contains(
        "Load factors - for schemes operating on an unchanged configuration basis",
        na=False
    )
].min()

df_63 = raw_63.iloc[header_idx + 1:next_section_idx].copy()
df_63.columns = raw_63.iloc[header_idx].tolist()
df_63 = df_63.rename(columns={df_63.columns[0]: "source_tech"})

rename_year_cols = {}
year_cols = []

for col in df_63.columns:
    clean = str(col).split("[")[0].replace(".0", "").strip()
    if clean.isdigit():
        y = int(clean)
        rename_year_cols[col] = y
        if 2010 <= y <= 2024:
            year_cols.append(y)

df_63 = df_63.rename(columns=rename_year_cols)

source_to_tech = {
    "Wind": "wind_total",
    "Solar photovoltaics": "solar",
    "Hydro": "hydro",
    "Bioenergy (excludes cofiring, includes non-biodegradable wastes)": "biomass",
}

df_63 = df_63[
    df_63["source_tech"].astype(str).str.strip().isin(source_to_tech.keys())
].copy()

df_63_long = df_63.melt(
    id_vars=["source_tech"],
    value_vars=year_cols,
    var_name="year",
    value_name="load_factor_pct"
)

df_63_long["year"] = df_63_long["year"].astype(int)
df_63_long["load_factor_pct"] = pd.to_numeric(df_63_long["load_factor_pct"], errors="coerce")
df_63_long["source_tech"] = df_63_long["source_tech"].astype(str).str.strip()
df_63_long["tech"] = df_63_long["source_tech"].map(source_to_tech)

df_63_final = (
    df_63_long
    .dropna(subset=["load_factor_pct", "tech"])
    .groupby(["year", "tech"], as_index=False)
    .agg(
        load_factor_pct=("load_factor_pct", "mean"),
        source_table=("source_tech", lambda s: "DUKES 6.3: " + "; ".join(sorted(set(s))))
    )
)

df_63_final = df_63_final[["year", "tech", "load_factor_pct", "source_table"]]
df_63_final = df_63_final.sort_values(["year", "tech"]).reset_index(drop=True)

assert not df_63_final.duplicated(["year", "tech"]).any()

print("6.3 extraction complete")
print("Rows:", len(df_63_final))
print("Duplicate year+tech rows:", df_63_final.duplicated(["year", "tech"]).sum())
print("Tech:", sorted(df_63_final["tech"].unique()))

In [ ]:
import pandas as pd

df_capacity = pd.read_parquet("output/dukes_capacity_hist_2010_2024.parquet")

assert not df_capacity.duplicated(["year", "tech"]).any(), (
    "dukes_capacity_hist_2010_2024.parquet still has duplicate year+tech rows"
)

print("Capacity deliverable left unchanged.")
print("Rows:", len(df_capacity))
print("Duplicate year+tech rows:", df_capacity.duplicated(["year", "tech"]).sum())
print("Tech:", sorted(df_capacity["tech"].unique()))

In [ ]:
import pandas as pd

df_510b = pd.read_parquet("output/dukes_loadfactor_hist_2010_2024.parquet")

overlap = set(df_510b["tech"]) & set(df_63_final["tech"])
assert not overlap, f"Unexpected overlap between 5.10B and 6.3 tech categories: {sorted(overlap)}"

df_loadfactor_combined = pd.concat([df_510b, df_63_final], ignore_index=True)

df_loadfactor_combined = (
    df_loadfactor_combined
    .groupby(["year", "tech"], as_index=False)
    .agg(
        load_factor_pct=("load_factor_pct", "mean"),
        source_table=("source_table", lambda s: "; ".join(sorted(set(s.astype(str)))))
    )
)

df_loadfactor_combined = df_loadfactor_combined[
    ["year", "tech", "load_factor_pct", "source_table"]
].sort_values(["year", "tech"]).reset_index(drop=True)

assert not df_loadfactor_combined.duplicated(["year", "tech"]).any()

df_loadfactor_combined.to_parquet(
    "output/dukes_loadfactor_hist_2010_2024.parquet",
    index=False
)

print("6.3 successfully combined with 5.10B load factors")
print("Rows:", len(df_loadfactor_combined))
print("Duplicate year+tech rows:", df_loadfactor_combined.duplicated(["year", "tech"]).sum())
print("Tech:", sorted(df_loadfactor_combined["tech"].unique()))

In [ ]:
import pandas as pd

df = pd.read_parquet("output/dukes_capacity_hist_2010_2024.parquet")

print("Total rows:", len(df))

print("\nData sources:")
print(df["source_table"].value_counts())

print("\nAll tech types:")
tech_list = sorted(df["tech"].unique())
for t in tech_list:
    print("-", t)

print("\nYear range:", df["year"].min(), "~", df["year"].max())

print("\nFirst 5 rows:")
print(df.head())

print("\nLast 5 rows:")
print(df.tail())

In [ ]:
import pandas as pd

df = pd.read_parquet("output/dukes_loadfactor_hist_2010_2024.parquet")

print("Total rows:", len(df))

print("\nData sources:")
print(df["source_table"].value_counts())

print("\nAll tech types:")
tech_list = sorted(df["tech"].unique())
for t in tech_list:
    print("-", t)

print("\nYear range:", df["year"].min(), "~", df["year"].max())

print("\nColumns:", df.columns.tolist())